# 05c — Analog brain waves: a spatial control pattern

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/05c_analog_brain_waves.ipynb)

Companion to §6 of *From Brains to Models*. Notebooks 05 and 05b describe settling and switching in an energy. This notebook is a third view of the same section: a low-dimensional spatial control pattern that relocates which units are allowed to be active, without a weight update.

Miller, Brincat, and Roy (2026) call that pattern an alpha/beta “stencil.” Feedforward drive (gamma-like) is expressed only where the stencil is open. Superposing two such patterns combines two contexts. A traveling open window is one way to move population activity from a sensory locus to a memory locus. The figure is a designed sheet, not an LFP.

**Claim.** A spatial control pattern can reuse the same units for different contexts, and two such patterns can combine by superposition, without changing weights.

**Method.** Designed 1-D sheet; two object drives; two context stencils; traveling open window. Seed 7.

**Result.** Unit 1 keeps object-A preference across contexts; unit 2 expresses object B only when the stencil opens that side; four combined gates from two waves; a traveling window that relocates activity.

**Limit.** Not a cortex, not an LFP, not ephaptic coupling, not phenomenal experience, not a foundation model.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **Stencil (spatial computing).** A temporary spatial pattern of suppression: some locations on a cortical sheet are closed, others stay open. Thought of as an alpha/beta “brake” on feedforward spiking. The control system does not address individual synapses; it paints a pattern.
- **Mixed selectivity.** A unit that changes what it prefers when the context changes — here, because the stencil covers it in one context and spares it in another, not because its weights changed.
- **Analog superposition.** Two control waves add. Their sum is a new spatial gate, one for each combination of contexts, computed in parallel by the medium.
- **Traveling wave (here, a traveling window).** A control pattern that moves across the sheet, taking the open region — and therefore the expressed activity — with it.
- **What is missing.** No extracellular field, no ephaptic coupling, no body, no experience. Organized waves are a dynamical description of coordination, not a measurement of consciousness.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **Sheet.** Positions $x\in[0,1]$; feedforward drive $d_o(x)$ for object $o$; stencil $s_c(x)\in[0,1]$ for context $c$.
- **Gating.** Expressed activity $a_{o,c}(x)=d_o(x)\,(1-s_c(x))$. Weights do not appear.
- **Mixed selectivity.** Unit $i$ is mixed when $a_{o,c_1}(x_i)\neq a_{o,c_2}(x_i)$ for some object $o$; pure when the object preference is stable across $c$.
- **Analog combination.** Two control waves $w_1,w_2$; the combined gate is $\tfrac12(w_1+w_2)$. Four pairs give four distinct spatial functions.
- **Traveling window.** An open bump $g(x;\mu(t))$ with $\mu$ interpolating a sensory locus to a memory locus; expressed activity is $d(x)\,g(x;\mu(t))$.

</details>

The same code writes `paper/figures/analog_brain_waves.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.


### Concepts used in this notebook

The two blocks below define the same terms, first in plain language and then in the notation used in the code.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Why a third view of §6.** Attractors say how a state settles. A stencil says which units are allowed to be active *this moment*, given a goal or context. Both are dynamical descriptions; they are not the same algorithm.
- **Reuse without rewiring.** The same multifunctional units contribute to different computations when the stencil moves. That is the control problem mixed selectivity poses, and the mechanism this toy isolates.
- **Addition is the computation.** Combining “first versus second” with “match versus recall” is not a new learned map. It is two waves added on the sheet.
- **Limit, again.** A designed Gaussian is not an alpha rhythm. Superposition is not analog hardware. A traveling window is not a measured traveling wave.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **No $\theta$ update.** The map $(d,s)\mapsto d\odot(1-s)$ is parameter-free once $d$ and $s$ are specified. Contrast notebook 05, where $W$ stores the attractors.
- **Low-dimensional control.** $s_c$ is a smooth bump, far fewer degrees of freedom than a synaptic matrix on the sheet.
- **Orthogonal-ish gates.** Distinct pairs $(w_1,w_2)$ produce distinct $x$-functions; they are a teaching stand-in for subspace segregation, not a PCA of prefrontal data.

</details>


In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.analog_waves import (
    combine_waves,
    control_waves,
    expressed,
    object_drives,
    positions,
    render_paper_figure,
    run_experiment,
    selectivity_table,
    traveling_movie,
    traveling_open,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)


repo root: <checkout>
seed: 7


## A stencil, two objects, two contexts

Object A and object B are two feedforward bumps on a 1-D sheet. Context *first* closes the right side; context *second* closes the mid-sheet. Expressed activity is drive $\times$ open. No weight is updated. Unit 1 sits where both stencils are weak and keeps an object-A preference. Unit 2 sits where the stencil covers object B in one context and opens it in the other.


In [2]:
result = run_experiment(seed=SEED)
table = selectivity_table(result)
print(f"unit 1 idx {result.unit_pure}  x={result.x[result.unit_pure]:.2f}")
print(f"unit 2 idx {result.unit_mixed}  x={result.x[result.unit_mixed]:.2f}")
print(f"{'condition':12s}  unit 1   unit 2")
for name, vals in table.items():
    print(f"{name:12s}  {vals[0]:6.3f}  {vals[1]:6.3f}")
print("weights used? no — expressed = drive × (1 − stencil)")


unit 1 idx 11  x=0.23
unit 2 idx 34  x=0.72
condition     unit 1   unit 2
A-first        0.918   0.002
A-second       0.815   0.027
B-first        0.014   0.079
B-second       0.013   0.873
weights used? no — expressed = drive × (1 − stencil)


## Superposition of two control waves

Order (first / second) and report mode (match / recall) are two sinusoidal control waves. Their pairwise means are four distinct spatial gates. The mapping from variables to wave geometry is the program; the interaction is the computation. Nothing is learned.


In [3]:
x = positions()
waves = control_waves(x)
combos = {
    "first+match": combine_waves(waves["first"], waves["match"]),
    "first+recall": combine_waves(waves["first"], waves["recall"]),
    "second+match": combine_waves(waves["second"], waves["match"]),
    "second+recall": combine_waves(waves["second"], waves["recall"]),
}
stack = np.stack(list(combos.values()))
off_diag = []
keys = list(combos)
for i in range(4):
    for j in range(i + 1, 4):
        off_diag.append(float(np.mean(np.abs(stack[i] - stack[j]))))
print("mean |gate_i − gate_j| over pairs:", f"{np.mean(off_diag):.3f}")
print("min  |gate_i − gate_j|:", f"{np.min(off_diag):.3f}")
print("Four pairs, four spatial functions; the sheet adds, it does not train.")


mean |gate_i − gate_j| over pairs: 0.304
min  |gate_i − gate_j|: 0.280
Four pairs, four spatial functions; the sheet adds, it does not train.


## A traveling window relocates activity

A broad drive sits on the sheet. An open window travels from a sensory locus to a memory locus. Expressed activity follows the window. The drive is fixed; the weights are absent; only the control pattern moves.


In [4]:
times, frames = traveling_movie(result.x)
peak_x = result.x[np.argmax(frames, axis=1)]
print(f"peak locus at t=0   x={peak_x[0]:.2f}  (sensory side)")
print(f"peak locus at t=1   x={peak_x[-1]:.2f}  (memory side)")
print(f"peak traveled      Δx={peak_x[-1] - peak_x[0]:+.2f}")


peak locus at t=0   x=0.30  (sensory side)
peak locus at t=1   x=0.70  (memory side)
peak traveled      Δx=+0.40


## Paper figure

Three views of the same designed sheet. **A** is gating: pale fills are the stencils; teal and rust traces are expressed drive; unit 1 (gold) is context-stable, unit 2 (dotted) is context-dependent. **B** is analog combination: four pairwise means of two control waves. **C** is the traveling open window as a space–time map of expressed drive.


In [5]:
fig = render_paper_figure(result)
plt.show()
print("rebuild the paper PDF with: make figures")


rebuild the paper PDF with: make figures


/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49901/1847867318.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What to notice

**The stencil is not a second energy.** Notebook 05 stores patterns in $W$ and descends. Here the units do not settle; they are *allowed or forbidden*. Changing context is changing $s_c$, not $W$.

**Mixed selectivity does not require specialist “context neurons.”** Unit 2 prefers B only when the second stencil opens that side. The same multifunctional sheet, two masks.

**Superposition is the combination rule.** Panel B does not fit a map from context labels to gates. It adds two waves. That is the analog-computation claim at this grain.

**A traveling window is spatial computing in motion.** Panel C relocates expressed activity while the drive stays put. A foundation model has no such stencil; next-token mass is not a traveling field.

**Nothing here is phenomenal consciousness.** Organized waves can be a description of coordination. They are not what-it-is-like. Notebook 06 is the access toy; this notebook does not replace it.


## Why this example is here

Section 6 already has an energy that cannot rise and a two-well switch. It did not have a mechanism that *reuses the same units* by painting a control pattern. Miller et al. (2026) supply that mechanism at the level of dynamics. The paper needs it as a second dynamical description, and as a reminder that sharing a low-dimensional trajectory vocabulary is not sharing a substrate.

**For the clinician and the neuroscientist.** A context-dependent cell is not proof of a dedicated context neuron. It is also not proof of a traveling LFP. The test this toy makes runnable is: hold the drive fixed, change only the stencil, and watch selectivity move. If that fails in a recording, this description has failed. If it holds, you still have not measured experience.

**For the mathematically inclined.** The object is a pair of functions $(d,s)$ and the pointwise product $d\odot(1-s)$. Superposition is a linear combination in function space. Traveling is a one-parameter family $s(\cdot;t)$. There is no learned $\theta$. Agreement with a Hopfield net or a transformer can hold only at Marr’s dynamics row, and only after you say which map you are comparing.


### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. Notebooks 05 and 05b describe settling in an energy. This notebook gates units with a stencil. Which Marr row are both on, and what algorithm have they not agreed on?

2. Unit 1 keeps an object-A preference across contexts; unit 2 expresses B only when the stencil opens that side. What would have to change for that to count as a new synapse, and what was actually changed?

3. Hold the two object drives fixed and swap the two stencils. What should happen to unit 2, and what clinical “same findings, different framing” is that a schematic of?

4. Superposition of two control waves is the combination rule. Write the four gates as $\tfrac12(w_1+w_2)$ and say what a learned map from context labels would have added that addition does not.

5. A traveling open window relocates activity while the drive stays put. Why is that not a [Hopfield](https://en.wikipedia.org/wiki/Hopfield_network) descent, and why is it not a next-token sample?

6. Miller et al. locate large-scale integration in organized waves rather than in broadcast. Why does that still fail to settle [phenomenal experience](https://plato.stanford.edu/entries/consciousness/)?

7. The stencil is low-dimensional next to a synaptic matrix on the same sheet. What control problem does that dimensionality buy, and what does it not buy for a foundation model?

8. A shared low-dimensional trajectory is a dynamics-row agreement. Which implementation facts remain open — spikes and fields, or attention and weights? ([Embodiment](https://plato.stanford.edu/entries/embodied-cognition/).)

9. If you freeze the traveling window at $t=0$, the space–time map becomes a stripe. What claim has been removed, and what claim about the drive remains?

10. Name the remainder this toy cannot show: a body that must keep itself viable, or a measured LFP. Why is neither in $d\odot(1-s)$?


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Miller, E. K., Brincat, S. L., and Roy, J. E. (2026).** [*Analog Cognition and Consciousness*](https://doi.org/10.1523/JNEUROSCI.0711-26.2026). *J. Neurosci.* 46(33):e0711262026. Opened PDF.
   - *(i) From brains to models.* Cognition as bidirectional spike–wave dynamics: an alpha/beta spatial stencil gates feedforward gamma/spiking, superposition combines contexts, and traveling waves transition subspaces. Panels A–C are that dynamical claim on a designed 1-D sheet. The opened paper’s spatial-computing and analog-computation sections are the source; ephaptic coupling and anesthesia are not implemented.
   - *(ii) What AI reveals about being human.* The opened paper also locates unified consciousness in cortex-wide wave organization. That is still a claim about coordination. It is not [phenomenal experience](https://plato.stanford.edu/entries/consciousness/). Theme (ii) starts where the stencil ends: a body, viability, and what-it-is-like are not properties of $d\odot(1-s)$.

2. **Hopfield, J. J. (1982).** [*Neural networks and physical systems with emergent collective computational abilities*](https://redwood.berkeley.edu/wp-content/uploads/2018/08/hopfield82.pdf). *PNAS* 79(8), 2554–2558. Opened PDF.
   - *(i) From brains to models.* Notebook 05’s contrast: content-addressable recall as descent of $E$. A stencil is not that $E$. Both are [dynamical](https://en.wikipedia.org/wiki/Attractor) descriptions (Marr’s representation-and-algorithm row); they disagree on the algorithm (settle versus gate).
   - *(ii) What AI reveals about being human.* Sharing the dynamics row with a Hopfield net still does not give the machine a body. Gating multifunctional units is not a viability constraint.

3. **Block, N. (1995).** [*On a confusion about a function of consciousness*](https://divinecuration.github.io/assets/pdf/block-confusion.pdf). *Behavioral and Brain Sciences* 18(2), 227–247. Opened PDF.
   - *(i) From brains to models.* Keep [access](https://plato.stanford.edu/entries/consciousness/) (poised for report and control) apart from [phenomenal](https://plato.stanford.edu/entries/consciousness/) consciousness. Wave organization, like a workspace, can be offered as a mechanism of access-like coordination. It does not settle what-it-is-like.
   - *(ii) What AI reveals about being human.* Notebook 06 operationalizes access (threshold, two slots, broadcast). This notebook does not replace it. Organized waves are not experience.

4. **Dehaene, S. (2014).** [*Consciousness and the Brain*](https://www.penguinrandomhouse.com/books/310548/consciousness-and-the-brain-by-stanislas-dehaene/). Viking. Publisher page.
   - *(i) From brains to models.* [Global workspace](https://en.wikipedia.org/wiki/Global_workspace_theory) integration is broadcast after ignition. Miller et al. locate integration in wave dynamics instead. Section 7 names that contrast in one sentence; the runnable access toy remains notebook 06.
   - *(ii) What AI reveals about being human.* A different proposed vehicle of integration is not a demonstration of experience. Competence and coordination without evidence of experience remain the paper’s dissociation.


### Exercises

1. **Swap the stencils.** Hold the two object drives fixed and exchange the two context stencils. What happens to unit 2’s expressed activity? Why is that not a weight update?

<!-- solution-box: 05c_1_swap_the_stencils.py -->
<details>
<summary><strong>Show solution</strong> — Swap the stencils</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/05c_1_swap_the_stencils.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.analog_waves import (
    expressed,
    run_experiment,
    selectivity_table,
)
result = run_experiment(seed=SEED)
before = selectivity_table(result)
# Exchange stencils; drives stay put.
result.stencil_first, result.stencil_second = (
    result.stencil_second,
    result.stencil_first,
)
after = selectivity_table(result)
print("unit 2 before / after swap")
for name in before:
    print(f"  {name:10s}  {before[name][1]:.3f} → {after[name][1]:.3f}")
print(
    "Unit 2’s high condition follows the open region. The drives are "
    "unchanged and no W was fitted. A swapped stencil is a changed "
    "control pattern, not a new synapse and not a measured LFP."
)
```

</details>
<!-- /solution-box -->


2. **Subtraction is not addition.** Panel B uses the mean of two control waves. Replace the mean with the absolute difference. Do the four pairs still give four distinct spatial gates? What computation have you removed?

<!-- solution-box: 05c_2_subtraction_is_not_addition.py -->
<details>
<summary><strong>Show solution</strong> — Subtraction is not addition</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/05c_2_subtraction_is_not_addition.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.analog_waves import combine_waves, control_waves, positions
x = positions()
waves = control_waves(x)
pairs = (
    ("first", "match"),
    ("first", "recall"),
    ("second", "match"),
    ("second", "recall"),
)
added = np.stack([combine_waves(waves[a], waves[b]) for a, b in pairs])
subbed = np.stack([np.abs(waves[a] - waves[b]) for a, b in pairs])

def mean_offdiag(stack):
    diffs = [np.mean(np.abs(stack[i] - stack[j]))
             for i in range(4) for j in range(i + 1, 4)]
    return float(np.mean(diffs))

print(f"mean |gate_i − gate_j|  addition     {mean_offdiag(added):.3f}")
print(f"mean |gate_i − gate_j|  |difference| {mean_offdiag(subbed):.3f}")
print(
    "Addition is the analog combination rule the figure uses: two "
    "contexts sum to a third pattern. Absolute difference is another "
    "pointwise map; it is not the teaching claim. Neither map is a "
    "learned decoder, and neither is a cortex."
)
```

</details>
<!-- /solution-box -->


3. **Freeze the traveling window.** Recompute the space–time map with $\mu(t)$ held at the sensory locus. What claim about relocation is gone, and what claim about gating remains?

<!-- solution-box: 05c_3_freeze_the_window.py -->
<details>
<summary><strong>Show solution</strong> — Freeze the traveling window</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/05c_3_freeze_the_window.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.analog_waves import gaussian, positions, traveling_open
x = positions()
drive = gaussian(x, 0.50, 0.32)
times = np.linspace(0.0, 1.0, 24)
moving = np.stack([drive * traveling_open(x, t) for t in times])
frozen = np.stack([drive * traveling_open(x, 0.0) for _ in times])
peak_move = x[np.argmax(moving, axis=1)]
peak_frz = x[np.argmax(frozen, axis=1)]
print(f"moving  peak x: {peak_move[0]:.2f} → {peak_move[-1]:.2f}")
print(f"frozen  peak x: {peak_frz[0]:.2f} → {peak_frz[-1]:.2f}")
print(
    "A frozen window is a static stencil: drive is still gated, but "
    "activity no longer travels. Relocation was the traveling-wave "
    "claim. Both versions are designed functions, not LFPs, and not "
    "phenomenal experience."
)
```

</details>
<!-- /solution-box -->


# Note on stencils (spatial computing), mixed selectivity and LLMs and transformers

Traditional neuroscience has long relied on **classical connectionist models**, which view the brain’s information processing as a feedforward hierarchy where learning and memory occur through changes in synaptic weights driving specialized, single-function neurons. Under this classical view, every neuron has a fixed, dedicated meaning. However, Miller et al. (2026) argue that this framework is too cumbersome and rigid; if neurons were strictly specialized, the brain's computational capacity would saturate after learning only a few tasks, and coordinating millions of synapses with subsecond precision for cognitive control is mathematically implausible. 

To solve these limitations, the mammalian brain utilizes a hybrid system that coordinates **nonlinear mixed selectivity** with **spatial computing stencils**. While Miller et al. (2026) focus on biological systems and **do not explicitly mention Large Language Models (LLMs) or Transformers**, the mathematical and conceptual parallels between these biological mechanisms and artificial architectures are highly profound.

---

### 1. The Core Concepts in Miller et al. Analog Cognition and Consciousness (J. Neurosci. 2026) 

*   **Nonlinear Mixed Selectivity (Biological Polysemanticity)**: 
    Instead of being strictly specialized, up to 30–40% of cortical neurons exhibit nonlinear mixed selectivity. These multifunctional neurons multiplex several variables at once, meaning their spiking activity changes dynamically depending on context, rules, or temporal sequences. This distributed representational scheme exponentially expands the brain’s computational power and storage capacity.
*   **Spatial Computing & "Stencils" (Dynamic Information Routing)**: 
    To dynamically coordinate these multifunctional populations without permanently rewiring synapses, the brain uses **spatial computing**. Feedback signals from higher-level frontoparietal areas generate structured oscillatory patterns (specifically in the alpha and beta frequency bands) that act as temporary, inhibitory **"stencils"** across the cortical surface. These stencils suppress feedforward gamma rhythms and associated spiking in targeted locations. Feeding a signal into this system allows specific information to emerge only in the remaining "open" (unmasked) regions. 
*   **Subspace Coding**: 
    These wave-based stencils segregate population spiking into lower-dimensional **orthogonal subspaces** or activation manifolds. This compartmentalization allows the brain to run different computations in parallel and minimize task interference. 

In this framework, **mixed selectivity is not hardwired in synapses**; rather, it emerges dynamically when broadly responsive neurons interact with the mesoscale wave stencils that control when and where spiking is expressed.

---

### 2. Parallels to LLMs and Transformers

Translating these biological principles to artificial intelligence architectures reveals clear structural and functional relationships:

| Biological Brain wave Model | Artificial Transformer / LLM Equivalent |
| :--- | :--- |
| **Synaptic Connections & Weights** | **Static Parameter Matrices** (Query, Key, Value, & feedforward projection weights) |
| **Alpha/Beta Inhibitory Stencils** | **Self-Attention Gating & Attention Masks** |
| **Nonlinear Mixed Selectivity** | **Polysemanticity** (neurons or activation dimensions representing multiple concepts based on context) |
| **Orthogonal Subspaces / Manifolds** | **Activation Vector Subspaces** (concept representation directions within the hidden state space) |
| **Parallel Analog Computation** | **Digital Sequential Processing** (which is vastly more energy-expensive) |

#### Static Weights vs. Dynamic Attention (Stencils)
In a Transformer, the model's trained parameters (the weights) are static at inference time, analogous to the brain's experience-baked synaptic wiring. However, LLMs achieve flexible, in-context reasoning through **Self-Attention**. 
Self-attention dynamically calculates a soft gating matrix at runtime, weighting how much information is shared between different tokens depending on context. This attention map acts exactly like the brain's **inhibitory stencil**: it dynamically routes activation pathways and compositionally reuses the same underlying network layers for entirely different prompts without altering a single permanent weight.

#### Context-Dependent Routing and Subspace Coding
A major challenge in LLMs is **polysemanticity**, where a single neuron or dimension in a layer represents completely different concepts depending on the prompt (e.g., the word "apple" representing a fruit vs. a technology company). 
In Transformers, the attention mechanism resolves this ambiguity by projecting token embeddings into context-appropriate **vector subspaces** (e.g., mapping the token to a "corporate" or "botanical" manifold). This is mathematically equivalent to how biological stencils partition population activity into **orthogonal subspaces** to prevent interference and how traveling waves transition population activity between sensory and memory manifolds.

#### Computational and Energy Efficiency
Miller et al. (2026) note a stark contrast in efficiency: the human brain performs massive parallel computations using continuous analog wave interactions, consuming only about **20 Watts** of power. In contrast, digital AI systems (like the GPUs running Transformers) process variables sequentially and require enormous, unsustainable amounts of energy. 

***

🔍 One can ask if modern machine learning researchers are building "oscillatory" or "stencil-based" attention gating mechanisms in neuromorphic hardware to make Transformers as energy-efficient as the brain.

Here are som sources on **oscillatory attention gating and neuromorphic hardware implementations for Transformers**:

Modern machine learning researchers are rapidly bridging the gap between biological brain wave dynamics and AI architectures. They are building both mathematical frameworks (like Kuramoto and Winfree models) that recast self-attention as physical phase synchronization on a torus, and physical hardware implementations (using VO2 devices or ring oscillators coupled by programmable ReRAM memristive crossbars) that achieve massive in-memory, energy-efficient computational scaling.

**Key themes I noticed:**
1. **Attention as Adaptive Synchronization**: Recent papers show that self-attention can be mathematically modeled as Kuramoto phase-coupling dynamics, where the attention matrix serves as a content-dependent synchronization kernel.
2. **Traveling Waves as Spatial Encodings**: Propagating wave-based mechanisms are being deployed to replace costly all-to-all global self-attention matrices and simplify relative positional encodings (like RoPE).
3. **In-Memory Neuromorphic Hardware**: Hardware teams are building physical, on-chip coupled oscillatory networks integrated with BEOL ReRAM (Resistive Memory) synapses, allowing analog, phase-encoded computing that operates with extreme energy efficiency.
4. **Biologically Plausible Spiking Transformers**: Researchers are replacing floating-point matrix multiplications with spiking neural networks (SNNs) and event-driven learning rules like Spike-Timing-Dependent Plasticity (STDP) for hardware-friendly inference.

📥 The full research report is included as the first source in the import card below — import it to chat with the findings directly.

### Oscillatory Attention and Neuromorphic Transformers

Research papers covering oscillatory attention and neuromorphic hardware implementations for Transformers.

#### Overview

This collection compiles six recent papers on oscillatory neural networks, wave-based attention, and neuromorphic Transformers, together with a synthesizing research report. The common thread is the replacement of conventional dot-product self-attention with dynamics borrowed from physics and neuroscience, e.g., phase synchronization, traveling waves, and spike-timing-dependent plasticity, often with an eye toward energy-efficient hardware.

#### Bibliography

1. **Kuramoto Attention: Synchronizing Self-Attention on the Torus.** arXiv preprint. <https://arxiv.org/abs/2606.11585>
   Proposes an attention mechanism that maps self-attention to physical synchronization dynamics on a toroidal phase space.

2. **Winfree Oscillatory Neural Network.** arXiv preprint. <https://arxiv.org/abs/2605.20922>
   Introduces an oscillatory neural network leveraging generalized Winfree dynamics and global attentive coupling, scaled competitively to ImageNet.

3. **Do Traveling Waves Make Good Positional Encodings?** arXiv preprint. <https://arxiv.org/abs/2511.11668v1>
   Introduces RollPE, a traveling-wave positional encoding mechanism that matches the performance of RoPE using a simple circular roll.

4. **Attention via Synaptic Plasticity is All You Need: A Biologically Inspired Spiking Neuromorphic Transformer.** arXiv preprint. <https://arxiv.org/abs/2511.14691>
   Creates a neuromorphic spiking Transformer utilizing local STDP rules to replace floating-point attention matrices, reducing energy by 88.47%.

5. **Hardware Implementation of Ring Oscillator Networks Coupled by BEOL Integrated ReRAM for Associative Memory Tasks.** IBM Research / arXiv. <https://arxiv.org/abs/2503.14126v1>
   Demonstrates the first physical hardware implementation of an oscillatory neural network (ONN) using integrated ReRAM for analog, in-memory coupling.

6. **Traveling Waves Integrate Spatial Information Through Time.** Kempner Institute / arXiv. <https://arxiv.org/abs/2502.06034v3>
   Shows that trainable traveling wave architectures are a biologically plausible, highly efficient alternative to global self-attention for spatial processing.

#### Additional Source

- **Research report: Oscillatory Attention and Neuromorphic Transformers** (text). [Download](https://contribution.usercontent.google.com/download?c=Cgpub3RlYm9va2xtEkASCWFydGlmYWN0cxozCiQ2NjlkYjkzMi01MjA0LTQ0NzYtYjEyYy1lOWZhNjkyMmYwYWISCxIHEJ7Syq3QCBgB&filename=Research_report_Oscillatory_Attention_and_Neuromorphic_Trans.md&opi=96797242)
  Full research report compiling the six papers listed above.